# Reproducing the final-score submission (v1: private F1 0.89841, public 0.88966)

This notebook regenerates `submission_lgb_quick.csv` (repo copy: `submissions/v1_lgb_public_0.8897.csv`) **from the raw
competition data**. It is the code of `scripts/build_features.py` (final world) + `scripts/v1_quick_lgb.py` in one notebook:

1. decode the sensors and repair the dates
2. rebuild the minute-by-minute timeline (train + test interleaved by dynamic programming)
3. build test-like features for the training rows (10 out-of-fold chunks) and features for the test rows
4. train one LightGBM (1,400 rounds, seed 42) on all training rows
5. threshold 0.5 -> submission

The full method, all model comparisons and robustness checks are in `anomaly_detection.ipynb`.

**Run:** CPU is enough, Internet optional, about 30–45 min. Output: `/kaggle/working/submission.csv`. The last cell checks the
file against the submitted one (number of anomalies and MD5 fingerprint).

In [1]:
import os, sys, glob, time, hashlib, warnings
import numpy as np, pandas as pd, lightgbm as lgb, numba
warnings.filterwarnings('ignore')
t0 = time.time()
def log(*a): print(*a, f'[{(time.time() - t0) / 60:.1f} min]', flush=True)

hits = glob.glob('/kaggle/input/**/train.parquet', recursive=True) + glob.glob('data/train.parquet')
DATA = os.path.dirname(hits[0])
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.makedirs('src', exist_ok=True); sys.path.insert(0, 'src')
print('data:', DATA, '| output:', OUT)
print('lightgbm', lgb.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| numba', numba.__version__)
print('development run used lightgbm 4.7.0, numpy 2.5.3, pandas 3.0.5, numba 0.68.0')

data: data | output: .
lightgbm 4.7.0 | numpy 2.5.3 | pandas 3.0.5 | numba 0.68.0
development run used lightgbm 4.7.0, numpy 2.5.3, pandas 3.0.5, numba 0.68.0


## Library code (identical to `src/pipeline.py` and `src/softalign.py` in the repository)

In [2]:
%%writefile src/pipeline.py
"""Shared pipeline: decode sensors, fix dates, rebuild the minute timeline, build features."""
import numpy as np
import pandas as pd
import numba

SENS = ['x1', 'x2', 'x3', 'x4', 'x5']


def load(data_dir):
    tr = pd.read_parquet(f'{data_dir}/train.parquet')
    te = pd.read_parquet(f'{data_dir}/test.parquet')
    tr['target'] = tr['target'].astype(int)
    return tr, te


def decode(df):
    """Undo the exp/log transforms applied by the organisers (verified: the results are integer/regular grids)."""
    out = pd.DataFrame(index=df.index)
    out['x1'] = np.log(df['X1'])                       # 0 .. 1.39 on a 0.002 grid
    out['x2'] = np.exp(df['X2'])                       # ~224 .. 255  (a voltage-like reading)
    out['x3'] = np.round(np.log(df['X3'])).astype(int)  # integer count 0 .. 88
    out['x4'] = np.round(np.log(df['X4'])).astype(int)  # integer count 0 .. 80
    out['x5'] = np.round(np.exp(df['X5'])).astype(int)  # integer 1 .. 32
    return out


def fix_dates(df):
    """Days <= 12 have month/day swapped. File order is the true time order, so walk the day-blocks and
    pick the candidate (original or swapped) that keeps the calendar increasing."""
    blk = (df['Date'] != df['Date'].shift()).cumsum() - 1
    firsts = df['Date'].groupby(blk).first()
    prev, fixed = None, []
    for d in firsts:
        cands = [d]
        if d.day <= 12:
            try:
                cands.append(pd.Timestamp(year=d.year, month=d.day, day=d.month))
            except ValueError:
                pass
        if prev is not None:
            cands = [c for c in cands if c > prev] or cands
            d = min(cands, key=lambda c: abs((c - prev).days))
        fixed.append(d)
        prev = d
    return blk.values, pd.Series(fixed).values[blk.values]


@numba.njit(cache=True)
def _merge_two(A, B, w):
    """Optimal interleaving of two ordered sequences A, B that minimises the total weighted L1 jump
    between consecutive readings (sensors are smooth in time). Returns src[k] in {0: A, 1: B}."""
    n = A.shape[0]
    m = B.shape[0]
    INF = 1e18
    C = np.full((n + 1, m + 1, 2), INF)
    P = np.zeros((n + 1, m + 1, 2), np.int8)
    if n > 0:
        C[1, 0, 0] = 0.0
    if m > 0:
        C[0, 1, 1] = 0.0
    nf = A.shape[1]
    for i in range(n + 1):
        for j in range(m + 1):
            for k in range(2):
                c = C[i, j, k]
                if c >= INF:
                    continue
                if k == 0:
                    last = A[i - 1]
                else:
                    last = B[j - 1]
                if i < n:
                    v = c
                    for t in range(nf):
                        v += w[t] * abs(last[t] - A[i, t])
                    if v < C[i + 1, j, 0]:
                        C[i + 1, j, 0] = v
                        P[i + 1, j, 0] = k
                if j < m:
                    v = c
                    for t in range(nf):
                        v += w[t] * abs(last[t] - B[j, t])
                    if v < C[i, j + 1, 1]:
                        C[i, j + 1, 1] = v
                        P[i, j + 1, 1] = k
    k = 0 if C[n, m, 0] <= C[n, m, 1] else 1
    i, j, L = n, m, n + m
    src = np.empty(n + m, np.int8)
    while i + j > 0:
        L -= 1
        src[L] = k
        pk = P[i, j, k]
        if k == 0:
            i -= 1
        else:
            j -= 1
        k = pk
    return src


MERGE_W = np.array([1.0, 2.0, 0.5, 0.5, 1.0])


def merge_timeline(SA, dayA, SB, dayB):
    """SA/SB: standardised sensor matrices in their own file order; dayA/dayB: (sorted) day ids.
    Returns integer timeline positions for every row of A and B (one global ordering)."""
    posA = np.empty(len(SA), np.int64)
    posB = np.empty(len(SB), np.int64)
    days = np.union1d(dayA, dayB)
    ia0, ia1 = np.searchsorted(dayA, days, 'left'), np.searchsorted(dayA, days, 'right')
    ib0, ib1 = np.searchsorted(dayB, days, 'left'), np.searchsorted(dayB, days, 'right')
    off = 0
    for a0, a1, b0, b1 in zip(ia0, ia1, ib0, ib1):
        src = _merge_two(SA[a0:a1], SB[b0:b1], MERGE_W)
        k = np.arange(len(src)) + off
        posA[a0:a1] = k[src == 0]
        posB[b0:b1] = k[src == 1]
        off += len(src)
    return posA, posB


def standardise(*mats):
    allm = np.vstack(mats)
    mu, sd = allm.mean(0), allm.std(0)
    return [(m - mu) / sd for m in mats]


def sensor_matrix(dec):
    return dec[SENS].to_numpy(np.float64)


# ----------------------------------------------------------------------------- features
class _Store(dict):
    def __init__(self, keep): super().__init__(); self.keep = keep
    def __setitem__(self, k, v): super().__setitem__(k, np.asarray(v, dtype=np.float32)[self.keep])


def sensor_features(tl, keep):
    f = _Store(keep)
    for c in SENS:
        s = tl[c].astype(float)
        f[c] = s
        for k in (1, 2, 3):
            f[f'{c}_lag{k}'] = s.shift(k) - s
            f[f'{c}_lead{k}'] = s.shift(-k) - s
        for w in (5, 15, 61, 241):
            r = s.rolling(w, center=True, min_periods=1)
            m = r.mean()
            f[f'{c}_m{w}'] = m
            f[f'{c}_dev{w}'] = s - m
            f[f'{c}_s{w}'] = r.std()
            if w in (5, 15, 61):
                f[f'{c}_mx{w}'] = r.max()
                f[f'{c}_mn{w}'] = r.min()
        for w in (15, 61):
            f[f'{c}_pdev{w}'] = s - s.rolling(w, min_periods=1).mean()            # vs. past window
            f[f'{c}_fdev{w}'] = s - s[::-1].rolling(w, min_periods=1).mean()[::-1]  # vs. future window
        f[f'{c}_ewm'] = s - s.ewm(span=10).mean()
        g = s.groupby(tl['day'])
        dm, ds = g.transform('mean'), g.transform('std')
        f[f'{c}_daymean'] = dm
        f[f'{c}_daystd'] = ds
        f[f'{c}_dayz'] = (s - dm) / (ds + 1e-9)
        f[f'{c}_dayrank'] = g.rank(pct=True)
    x2 = tl['x2']
    for w in (15, 61):
        f[f'x2_drop_past{w}'] = x2.rolling(w, min_periods=1).max() - x2
        f[f'x2_drop_fut{w}'] = x2[::-1].rolling(w, min_periods=1).max()[::-1] - x2
    f['x2_absdiff_s15'] = x2.diff().abs().rolling(15, center=True, min_periods=1).mean()
    f['x3_pos'] = (tl['x3'] > 0).astype(int)
    f['x4_pos'] = (tl['x4'] > 0).astype(int)
    f['x34'] = tl['x3'] + tl['x4']
    f['x3_runlen'] = _run_len(tl['x3'].to_numpy() > 0)
    f['x4_runlen'] = _run_len(tl['x4'].to_numpy() > 0)
    f['x3_runlen_b'] = _run_len(tl['x3'].to_numpy()[::-1] > 0)[::-1]
    f['x4_runlen_b'] = _run_len(tl['x4'].to_numpy()[::-1] > 0)[::-1]
    is1 = (tl['x1'] == 0).astype(int); f['x1_is1'] = is1
    f['x1_is1_m61'] = is1.rolling(61, center=True, min_periods=1).mean()
    # calendar / trend
    dt = pd.to_datetime(tl['date'])
    f['minute'] = tl['minute']
    f['month'] = dt.dt.month
    f['doy'] = dt.dt.dayofyear
    f['dow'] = dt.dt.dayofweek
    f['t_days'] = (dt - dt.min()).dt.days
    f['day_n'] = tl.groupby('day')['x1'].transform('size')
    return pd.DataFrame(dict(f), index=tl.index[keep])


def _run_len(b):
    """Length of the current run of True values (0 where False)."""
    out = np.zeros(len(b), np.int64)
    run = 0
    for i, v in enumerate(b):
        run = run + 1 if v else 0
        out[i] = run
    return out


def label_features(pos_all, lab_pos, lab_y, day_all, lab_day):
    """Neighbour-label features from the *labelled* rows only, always excluding the row itself.
    pos_all: timeline position of every row we need features for; lab_pos/lab_y: positions & labels
    of the labelled rows (sorted by position)."""
    o = np.argsort(lab_pos)
    P, Y = lab_pos[o], lab_y[o].astype(float)
    CS = np.r_[0, np.cumsum(Y)]
    n = len(P)
    kb = np.searchsorted(P, pos_all, 'left')   # labelled rows strictly before
    ka = np.searchsorted(P, pos_all, 'right')  # first labelled row strictly after
    f = {}
    for k in (1, 2, 3, 5, 10, 30, 120):
        lo = np.maximum(kb - k, 0)
        hi = np.minimum(ka + k, n)
        f[f'yprev{k}'] = (CS[kb] - CS[lo]) / np.maximum(kb - lo, 1)
        f[f'ynext{k}'] = (CS[hi] - CS[ka]) / np.maximum(hi - ka, 1)
        f[f'yboth{k}'] = (CS[kb] - CS[lo] + CS[hi] - CS[ka]) / np.maximum(kb - lo + hi - ka, 1)
    # gap (timeline steps) to the nearest labelled neighbour on each side
    f['gap_prev'] = np.where(kb > 0, pos_all - P[np.maximum(kb - 1, 0)], 999)
    f['gap_next'] = np.where(ka < n, P[np.minimum(ka, n - 1)] - pos_all, 999)
    # distance to nearest labelled anomaly on each side
    AP = P[Y == 1]
    jb = np.searchsorted(AP, pos_all, 'left')
    ja = np.searchsorted(AP, pos_all, 'right')
    f['dist_prev_anom'] = np.where(jb > 0, pos_all - AP[np.maximum(jb - 1, 0)], 10**6).clip(max=5000)
    f['dist_next_anom'] = np.where(ja < len(AP), AP[np.minimum(ja, len(AP) - 1)] - pos_all, 10**6).clip(max=5000)
    f['dist_anom'] = np.minimum(f['dist_prev_anom'], f['dist_next_anom'])
    f = pd.DataFrame(f)
    # leave-self-out daily anomaly rate
    lab_df = pd.DataFrame({'day': lab_day, 'y': lab_y})
    dsum = lab_df.groupby('day')['y'].sum()
    dcnt = lab_df.groupby('day')['y'].size()
    s = pd.Series(day_all).map(dsum).fillna(0).to_numpy()
    c = pd.Series(day_all).map(dcnt).fillna(0).to_numpy()
    is_lab = np.isin(pos_all, P)
    # subtract self for labelled rows
    self_y = np.zeros(len(pos_all))
    idx = np.searchsorted(P, pos_all)
    idx = np.minimum(idx, n - 1)
    self_y[is_lab] = Y[idx[is_lab]]
    f['day_rate'] = (s - self_y) / np.maximum(c - is_lab, 1)
    return f


Overwriting src/pipeline.py


In [3]:
%%writefile src/softalign.py
"""Soft (posterior) alignment: forward-backward over the same interleaving lattice as the DP merge.
For every row of B we get P(i) = probability that exactly i rows of A precede it, and from it
expected neighbour-label features (expected label of previous/next labelled minutes, boundary prob...)."""
import numpy as np
import numba

KS = (1, 2, 3, 5, 10, 30)
SOFT_NAMES = ([f's_prev{k}' for k in KS] + [f's_next{k}' for k in KS] + [f's_both{k}' for k in KS] +
              ['s_inside', 's_boundary', 's_pmax', 's_entropy', 's_pos_std'])
NF = len(SOFT_NAMES)


@numba.njit(cache=True)
def _lae(a, b):
    if a == -np.inf:
        return b
    if b == -np.inf:
        return a
    if a > b:
        return a + np.log1p(np.exp(b - a))
    return b + np.log1p(np.exp(a - b))


@numba.njit(cache=True)
def _dist(x, y, w):
    s = 0.0
    for t in range(x.shape[0]):
        s += w[t] * abs(x[t] - y[t])
    return s


@numba.njit(cache=True)
def soft_day(A, B, yA, w, T):
    n, m = A.shape[0], B.shape[0]
    out = np.full((m, 23), np.nan)
    if m == 0 or n == 0:
        return out
    NI = -np.inf
    F = np.full((n + 1, m + 1, 2), NI)
    G = np.full((n + 1, m + 1, 2), NI)
    F[1, 0, 0] = 0.0
    F[0, 1, 1] = 0.0
    for i in range(n + 1):                      # forward (push)
        for j in range(m + 1):
            for k in range(2):
                v = F[i, j, k]
                if v == NI:
                    continue
                last = A[i - 1] if k == 0 else B[j - 1]
                if i < n:
                    F[i + 1, j, 0] = _lae(F[i + 1, j, 0], v - _dist(last, A[i], w) / T)
                if j < m:
                    F[i, j + 1, 1] = _lae(F[i, j + 1, 1], v - _dist(last, B[j], w) / T)
    G[n, m, 0] = 0.0
    G[n, m, 1] = 0.0
    for i in range(n, -1, -1):                  # backward
        for j in range(m, -1, -1):
            for k in range(2):
                if i == n and j == m:
                    continue
                if (k == 0 and i == 0) or (k == 1 and j == 0):
                    continue
                last = A[i - 1] if k == 0 else B[j - 1]
                v = NI
                if i < n:
                    v = _lae(v, G[i + 1, j, 0] - _dist(last, A[i], w) / T)
                if j < m:
                    v = _lae(v, G[i, j + 1, 1] - _dist(last, B[j], w) / T)
                G[i, j, k] = v
    logZ = _lae(F[n, m, 0], F[n, m, 1])
    CS = np.zeros(n + 1)
    for i in range(n):
        CS[i + 1] = CS[i] + yA[i]
    P = np.empty(n + 1)
    for j in range(m):
        tot = 0.0
        for i in range(n + 1):
            P[i] = np.exp(F[i, j + 1, 1] + G[i, j + 1, 1] - logZ)
            tot += P[i]
        if tot <= 0:
            continue
        for i in range(n + 1):
            P[i] /= tot
        acc = np.zeros(23)
        pmax = 0.0; ent = 0.0; mu = 0.0; m2 = 0.0
        for i in range(n + 1):
            p = P[i]
            if p < 1e-9:
                continue
            for q in range(6):
                k = KS[q]
                lo = max(i - k, 0); hi = min(i + k, n)
                pv = (CS[i] - CS[lo]) / max(i - lo, 1)
                nv = (CS[hi] - CS[i]) / max(hi - i, 1)
                bv = (CS[i] - CS[lo] + CS[hi] - CS[i]) / max(hi - lo, 1)
                acc[q] += p * pv; acc[6 + q] += p * nv; acc[12 + q] += p * bv
            yp = yA[i - 1] if i > 0 else 0.0
            yn = yA[i] if i < n else 0.0
            acc[18] += p * yp * yn                     # sits inside a labelled episode
            acc[19] += p * (yp + yn - 2 * yp * yn)     # sits on an episode boundary
            if p > pmax:
                pmax = p
            ent -= p * np.log(p)
            mu += p * i; m2 += p * i * i
        acc[20] = pmax; acc[21] = ent; acc[22] = np.sqrt(max(m2 - mu * mu, 0.0))
        out[j] = acc
    return out


def soft_features(SA, dayA, yA, SB, dayB, w, T):
    """SA/SB standardised sensors (day-sorted, own order), yA labels of A. Returns (len(SB), NF)."""
    out = np.full((len(SB), NF), np.nan, np.float32)
    days = np.union1d(dayA, dayB)
    ia0, ia1 = np.searchsorted(dayA, days, 'left'), np.searchsorted(dayA, days, 'right')
    ib0, ib1 = np.searchsorted(dayB, days, 'left'), np.searchsorted(dayB, days, 'right')
    for a0, a1, b0, b1 in zip(ia0, ia1, ib0, ib1):
        if b1 > b0:
            out[b0:b1] = soft_day(SA[a0:a1], SB[b0:b1], yA[a0:a1].astype(np.float64), w, T)
    return out


Overwriting src/softalign.py


## 1. Load, decode, repair dates, standardise

In [4]:
import pipeline
from pipeline import load, decode, fix_dates, sensor_matrix, standardise, merge_timeline, sensor_features, label_features
from softalign import soft_features, SOFT_NAMES

tr, te = load(DATA)
dtr, dte = decode(tr), decode(te)
dtr['day'], dtr['date'] = fix_dates(tr)
dte['day'], dte['date'] = fix_dates(te)
dtr['y'] = tr['target'].values
dtr['src'], dte['src'] = 'train', 'test'
dte['ID'] = te['ID'].values
Str, Ste = standardise(sensor_matrix(dtr), sensor_matrix(dte))
day_tr, day_te = dtr['day'].values, dte['day'].values
y_all = dtr['y'].values
log('loaded', dtr.shape, dte.shape, 'anomaly rate %.3f%%' % (100 * y_all.mean()))

loaded (1639424, 9) (409856, 9) anomaly rate 0.856% [0.0 min]


## 2–3. Timeline rebuild and features

`view(lab_idx, hidden_groups, want_idx)`: the training rows `lab_idx` keep their known order and labels; each hidden group is
re-positioned by the DP merge (exactly like test rows), then the real test rows are merged in. Features are computed for
`want_idx` (or for the test rows when `want_idx is None`). A row's own label is never used for its own features.

In [5]:
def view(lab_idx, hidden_groups, want_idx):
    seq = np.sort(lab_idx)
    for g in hidden_groups:
        g = np.sort(g)
        pa, pb = merge_timeline(Str[seq], day_tr[seq], Str[g], day_tr[g])
        allr = np.r_[seq, g]
        seq = allr[np.argsort(np.r_[pa, pb], kind='stable')]
    p1, p2 = merge_timeline(Str[seq], day_tr[seq], Ste, day_te)
    pos_tr = np.empty(len(dtr), np.int64)
    pos_tr[seq] = p1
    tl = pd.concat([dtr.assign(pos=pos_tr, row=np.arange(len(dtr))), dte.assign(pos=p2, row=-1)], ignore_index=True)
    tl['is_lab'] = False
    tl.loc[lab_idx, 'is_lab'] = True
    tl = tl.sort_values('pos').reset_index(drop=True)
    tl['minute'] = tl.groupby('day').cumcount() / tl.groupby('day')['x1'].transform('size') * 1440
    keep = np.isin(tl['row'].values, want_idx) if want_idx is not None else (tl['src'] == 'test').values
    F = sensor_features(tl, keep)
    lab = tl[tl.is_lab]
    LF = label_features(tl['pos'].values, lab['pos'].values, lab['y'].values, tl['day'].values, lab['day'].values)
    LF.index = tl.index
    return pd.concat([tl.loc[keep, ['row', 'pos', 'day', 'src', 'ID', 'y']], F, LF[keep].astype(np.float32)], axis=1)

K = 10
allrows = np.arange(len(dtr))
chunk = np.random.default_rng(7).integers(0, K, len(allrows))
parts = []
for k in range(K):
    ck = allrows[chunk == k]
    parts.append(view(allrows[chunk != k], [ck], ck).assign(chunk=k))
    log('training chunk', k)
trF = pd.concat(parts).sort_values('row').reset_index(drop=True); del parts
teF = view(allrows, [], None).reset_index(drop=True)
META = ['row', 'pos', 'day', 'src', 'ID', 'y', 'chunk']
base_cols = [c for c in trF.columns if c not in META]
log('features', trF.shape, teF.shape, len(base_cols), 'feature columns')

training chunk 0 [0.4 min]


training chunk 1 [0.8 min]


training chunk 2 [1.2 min]


training chunk 3 [1.6 min]


training chunk 4 [2.0 min]


...

training chunk 6 [2.8 min]


training chunk 7 [3.2 min]


training chunk 8 [3.6 min]


training chunk 9 [4.0 min]


features (1639424, 224) (409856, 223) 217 feature columns [4.5 min]


## 4. LightGBM on all training rows (exactly `scripts/v1_quick_lgb.py`)

In [6]:
P = dict(objective='binary', learning_rate=0.05, num_leaves=63, min_child_samples=50, feature_fraction=0.7,
         bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=4, seed=42)
model = lgb.train(P, lgb.Dataset(trF[base_cols], trF.y), 1400)
p = model.predict(teF[base_cols]); log('trained')
ids = te['ID'].values
out = pd.DataFrame({'ID': teF.ID.values, 'p': p}).set_index('ID').reindex(ids)
assert out.p.notna().all()
for t in (0.40, 0.45, 0.50, 0.55): print(t, 'pred rate', round((out.p >= t).mean() * 100, 4), '%')

trained [8.4 min]


0.4 pred rate 0.8364 %
0.45 pred rate 0.8274 %
0.5 pred rate 0.8157 %
0.55 pred rate 0.8069 %


## 5. Threshold 0.5 -> submission, and check against the submitted file

In [7]:
sub = pd.DataFrame({'ID': ids, 'target': (out.p.values >= 0.5).astype(int).astype(str)})
for f in ('submission.csv', 'submission_lgb_quick.csv'): sub.to_csv(f'{OUT}/{f}', index=False)
md5 = hashlib.md5(open(f'{OUT}/submission.csv', 'rb').read(), usedforsecurity=False).hexdigest()  # file fingerprint, not security
n1 = int((sub.target == '1').sum())
print(f'predicted anomalies: {n1} | submitted file: 3343')
print('MD5:', md5, '| submitted file: f1be2ec5d8e1055dc1efc7ff3ba4a05c')
print('IDENTICAL to the submitted file' if md5 == 'f1be2ec5d8e1055dc1efc7ff3ba4a05c' else
      'not byte-identical (usually different library versions) - compare row by row below')
ref = glob.glob('/kaggle/input/**/v1_lgb_public_0.8897.csv', recursive=True) + glob.glob('submissions/v1_lgb_public_0.8897.csv')
if ref:
    r = pd.read_csv(ref[0]).set_index('ID').target.reindex(ids).values
    print('rows differing from the submitted file:', int((sub.target.astype(int).values != r).sum()), 'of', len(r))
log('DONE')

predicted anomalies: 3343 | submitted file: 3343
MD5: f1be2ec5d8e1055dc1efc7ff3ba4a05c | submitted file: f1be2ec5d8e1055dc1efc7ff3ba4a05c
IDENTICAL to the submitted file
rows differing from the submitted file: 0 of 409856
DONE [8.4 min]
